#  IQL Robustness Demo: Training + Shift Evaluation + Video Recording
**CMPE 260 — Reinforcement Learning | Group 6**

This notebook runs the full demo pipeline and saves all outputs for class presentation:
1.  Train IQL agents offline (DoubleCritic 2Q vs TripleCritic 3Q)
2.  Evaluate robustness under 4 types of distribution shift
3.  Record MuJoCo agent videos (baseline vs shifted environments)
4.  Save all plots, results, and videos for offline presentation

**Self-contained:** This notebook includes all IQL, wrapper, and evaluation code inline — no external `iql/` package needed.

**Workflow:** Run this notebook once on Colab → download `demo_outputs.zip` → present in class

**Runtime:** ~30–40 min on Colab T4 GPU (training ~3–4 min each, evaluation ~5 min each)

In [ ]:
# Cell 1: Environment Setup (Colab vs Local)
import os, sys

# CRITICAL: Limit JAX GPU memory to 40% so MuJoCo rendering has room.
# Must be set BEFORE importing JAX. Without this, JAX grabs all 15GB
# on Colab T4 and MuJoCo rendering triggers an OOM kernel kill.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.4'

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Use Colab's pre-installed JAX (0.7+) — do NOT downgrade
    !pip install -q "flax>=0.10" "optax>=0.2" mujoco "gymnasium[mujoco]" \
        h5py ml_collections tensorboardX tensorflow-probability \
        imageio imageio-ffmpeg

    # Install OSMesa for CPU-based MuJoCo rendering (avoids GPU OOM)
    # Need both runtime lib and dev headers
    !apt-get install -y libosmesa6 libosmesa6-dev > /dev/null 2>&1
    os.environ['MUJOCO_GL'] = 'osmesa'  # CPU rendering — avoids GPU OOM
    print(f' MUJOCO_GL={os.environ["MUJOCO_GL"]} (CPU-only rendering)')

print(f" Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f" JAX GPU memory fraction: {os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION', 'default (100%)')}")

In [ ]:
# Cell 2: Verify GPU & Core Imports
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import csv, time, os, collections, copy, functools, pickle, urllib.request
from typing import Any, Callable, Dict, Optional, Sequence, Tuple
import imageio

import flax
import flax.linen as nn
import optax
import gymnasium as gym
from gymnasium.spaces import Box, Dict as GymDict

# ── JAX 0.7+ compatibility shim for TFP ──────────────────────
# TFP ≤0.25 accesses jax.interpreters.xla.pytype_aval_mappings
# at import time, but that was removed in JAX 0.7.  Patch it.
import jax.interpreters.xla as _xla
if not hasattr(_xla, 'pytype_aval_mappings'):
    _xla.pytype_aval_mappings = jax.core.pytype_aval_mappings
# ─────────────────────────────────────────────────────────────

from tensorflow_probability.substrates import jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

print(f"JAX version: {jax.__version__}")
print(f"Flax version: {flax.__version__}")
print(f"JAX devices: {jax.devices()}")
try:
    gpu_count = len(jax.devices('gpu'))
    print(f"GPU available: {gpu_count > 0} ({gpu_count} device(s))")
except RuntimeError:
    print("GPU available: False (CPU only)")

print(" All imports successful!")

In [ ]:
# Cell 3: JAX/Flax Compatibility Shims
# Handle API differences between JAX 0.4.x and JAX 0.7+

JAX_VERSION = tuple(int(x) for x in jax.__version__.split('.')[:2])

# jax.tree_map was deprecated in JAX 0.7+ in favor of jax.tree.map
if hasattr(jax, 'tree') and hasattr(jax.tree, 'map'):
    _tree_map = jax.tree.map
else:
    _tree_map = jax.tree_map

print(f"JAX version tuple: {JAX_VERSION}")
print(f"Using tree_map from: {'jax.tree.map' if hasattr(jax, 'tree') else 'jax.tree_map'}")

In [ ]:
# Cell 4: IQL Core — Model, MLP, Batch (replaces iql/common.py)
# This replaces flax.struct.dataclass with manual JAX pytree
# registration, which works across all JAX/Flax versions.

import os
import pickle
import collections
import numpy as np

Batch = collections.namedtuple(
    'Batch',
    ['observations', 'actions', 'rewards', 'masks', 'next_observations'])

PRNGKey = Any
Params = Any  # FrozenDict or plain dict depending on Flax version
Shape = Sequence[int]
Dtype = Any
InfoDict = Dict[str, float]


def default_init(scale: Optional[float] = jnp.sqrt(2)):
    return nn.initializers.orthogonal(scale)


class MLP(nn.Module):
    hidden_dims: Sequence[int]
    activations: Callable[[jnp.ndarray], jnp.ndarray] = nn.relu
    activate_final: int = False
    dropout_rate: Optional[float] = None

    @nn.compact
    def __call__(self, x: jnp.ndarray, training: bool = False) -> jnp.ndarray:
        for i, size in enumerate(self.hidden_dims):
            x = nn.Dense(size, kernel_init=default_init())(x)
            if i + 1 < len(self.hidden_dims) or self.activate_final:
                x = self.activations(x)
                if self.dropout_rate is not None:
                    x = nn.Dropout(rate=self.dropout_rate)(
                        x, deterministic=not training)
        return x


class Model:
    """Trainable model container — JAX pytree compatible.

    Replaces @flax.struct.dataclass which has compatibility issues
    across Flax versions. Uses manual pytree registration instead.
    """

    def __init__(self, step, apply_fn, params, tx=None, opt_state=None):
        self.step = step
        self.apply_fn = apply_fn  # NOT a pytree leaf
        self.params = params
        self.tx = tx              # NOT a pytree leaf
        self.opt_state = opt_state

    @classmethod
    def create(cls, model_def, inputs, tx=None):
        variables = model_def.init(*inputs)
        params = variables.get('params', variables)
        if isinstance(params, tuple):
            params = params[1]
        opt_state = tx.init(params) if tx is not None else None
        return cls(step=1, apply_fn=model_def, params=params,
                   tx=tx, opt_state=opt_state)

    def __call__(self, *args, **kwargs):
        return self.apply_fn.apply({'params': self.params}, *args, **kwargs)

    def apply(self, *args, **kwargs):
        return self.apply_fn.apply(*args, **kwargs)

    def apply_gradient(self, loss_fn):
        grad_fn = jax.grad(loss_fn, has_aux=True)
        grads, info = grad_fn(self.params)
        updates, new_opt_state = self.tx.update(grads, self.opt_state,
                                                self.params)
        new_params = optax.apply_updates(self.params, updates)
        return self.replace(step=self.step + 1, params=new_params,
                            opt_state=new_opt_state), info

    def replace(self, **kwargs):
        new = Model(
            step=kwargs.get('step', self.step),
            apply_fn=kwargs.get('apply_fn', self.apply_fn),
            params=kwargs.get('params', self.params),
            tx=kwargs.get('tx', self.tx),
            opt_state=kwargs.get('opt_state', self.opt_state),
        )
        return new

    def save(self, save_path):
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'wb') as f:
            pickle.dump(jax.device_get(self.params), f)

    def load(self, load_path):
        with open(load_path, 'rb') as f:
            params = pickle.load(f)
        return self.replace(params=params)


# Register Model as a JAX pytree so it can be used inside jit
def _model_flatten(model):
    # Children (traced by JAX): step, params, opt_state
    children = (model.step, model.params, model.opt_state)
    # Aux data (static, not traced): apply_fn, tx
    aux = (model.apply_fn, model.tx)
    return children, aux

def _model_unflatten(aux, children):
    step, params, opt_state = children
    apply_fn, tx = aux
    return Model(step=step, apply_fn=apply_fn, params=params,
                 tx=tx, opt_state=opt_state)

jax.tree_util.register_pytree_node(Model, _model_flatten, _model_unflatten)

print(" Model class registered as JAX pytree")

In [ ]:
# Cell 5: Policy Network (replaces iql/policy.py)
import functools
import numpy as np

LOG_STD_MIN = -10.0
LOG_STD_MAX = 2.0


class NormalTanhPolicy(nn.Module):
    hidden_dims: Sequence[int]
    action_dim: int
    state_dependent_std: bool = True
    dropout_rate: Optional[float] = None
    log_std_scale: float = 1.0
    log_std_min: Optional[float] = None
    log_std_max: Optional[float] = None
    tanh_squash_distribution: bool = True

    @nn.compact
    def __call__(self, observations, temperature=1.0, training=False):
        outputs = MLP(self.hidden_dims, activate_final=True,
                      dropout_rate=self.dropout_rate)(observations,
                                                      training=training)
        means = nn.Dense(self.action_dim, kernel_init=default_init())(outputs)

        if self.state_dependent_std:
            log_stds = nn.Dense(self.action_dim,
                                kernel_init=default_init(
                                    self.log_std_scale))(outputs)
        else:
            log_stds = self.param('log_stds', nn.initializers.zeros,
                                  (self.action_dim,))

        log_std_min = self.log_std_min or LOG_STD_MIN
        log_std_max = self.log_std_max or LOG_STD_MAX
        log_stds = jnp.clip(log_stds, log_std_min, log_std_max)

        if not self.tanh_squash_distribution:
            means = nn.tanh(means)

        base_dist = tfd.MultivariateNormalDiag(
            loc=means, scale_diag=jnp.exp(log_stds) * temperature)
        if self.tanh_squash_distribution:
            return tfd.TransformedDistribution(
                distribution=base_dist, bijector=tfb.Tanh())
        else:
            return base_dist


@functools.partial(jax.jit, static_argnames=('actor_def',))
def _sample_actions(rng, actor_def, actor_params, observations,
                    temperature=1.0):
    dist = actor_def.apply({'params': actor_params}, observations, temperature)
    rng, key = jax.random.split(rng)
    return rng, dist.sample(seed=key)


def sample_actions(rng, actor_def, actor_params, observations,
                   temperature=1.0):
    return _sample_actions(rng, actor_def, actor_params, observations,
                           temperature)

print(" Policy network defined")

In [ ]:
# Cell 6: Value Networks (replaces iql/value_net.py)
import numpy as np

class ValueCritic(nn.Module):
    hidden_dims: Sequence[int]

    @nn.compact
    def __call__(self, observations):
        critic = MLP((*self.hidden_dims, 1))(observations)
        return jnp.squeeze(critic, -1)


class Critic(nn.Module):
    hidden_dims: Sequence[int]
    activations: Callable[[jnp.ndarray], jnp.ndarray] = nn.relu

    @nn.compact
    def __call__(self, observations, actions):
        inputs = jnp.concatenate([observations, actions], -1)
        critic = MLP((*self.hidden_dims, 1),
                     activations=self.activations)(inputs)
        return jnp.squeeze(critic, -1)


class DoubleCritic(nn.Module):
    hidden_dims: Sequence[int]
    activations: Callable[[jnp.ndarray], jnp.ndarray] = nn.relu

    @nn.compact
    def __call__(self, observations, actions):
        critic1 = Critic(self.hidden_dims,
                         activations=self.activations)(observations, actions)
        critic2 = Critic(self.hidden_dims,
                         activations=self.activations)(observations, actions)
        return critic1, critic2


class TripleCritic(nn.Module):
    """Q-ensemble with 3 critics for more conservative value estimation."""
    hidden_dims: Sequence[int]
    activations: Callable[[jnp.ndarray], jnp.ndarray] = nn.relu

    @nn.compact
    def __call__(self, observations, actions):
        critic1 = Critic(self.hidden_dims,
                         activations=self.activations)(observations, actions)
        critic2 = Critic(self.hidden_dims,
                         activations=self.activations)(observations, actions)
        critic3 = Critic(self.hidden_dims,
                         activations=self.activations)(observations, actions)
        return critic1, critic2, critic3

print(" Value networks defined")

In [ ]:
# Cell 7: Critic Update Functions (replaces iql/critic.py)
import numpy as np

def expectile_loss(diff, expectile=0.8):
    weight = jnp.where(diff > 0, expectile, (1 - expectile))
    return weight * (diff ** 2)


def _min_q(critic_output):
    """Take element-wise min across all Q-networks (supports 2 or 3)."""
    if len(critic_output) == 2:
        return jnp.minimum(critic_output[0], critic_output[1])
    elif len(critic_output) == 3:
        return jnp.minimum(jnp.minimum(critic_output[0], critic_output[1]),
                           critic_output[2])
    else:
        raise ValueError(f"Expected 2 or 3 Q-networks, got {len(critic_output)}")


def update_v(critic, value, batch, expectile):
    actions = batch.actions
    critic_output = critic(batch.observations, actions)
    q = _min_q(critic_output)

    def value_loss_fn(value_params):
        v = value.apply({'params': value_params}, batch.observations)
        value_loss = expectile_loss(q - v, expectile).mean()
        return value_loss, {'value_loss': value_loss, 'v': v.mean()}

    new_value, info = value.apply_gradient(value_loss_fn)
    return new_value, info


def update_q(critic, target_value, batch, discount):
    next_v = target_value(batch.next_observations)
    target_q = batch.rewards + discount * batch.masks * next_v

    def critic_loss_fn(critic_params):
        critic_output = critic.apply({'params': critic_params},
                                     batch.observations, batch.actions)
        critic_loss = sum(
            (q - target_q) ** 2 for q in critic_output
        ).mean()
        info = {'critic_loss': critic_loss}
        for i, q in enumerate(critic_output):
            info[f'q{i+1}'] = q.mean()
        return critic_loss, info

    new_critic, info = critic.apply_gradient(critic_loss_fn)
    return new_critic, info

print(" Critic update functions defined")

In [ ]:
# Cell 8: Actor Update (replaces iql/actor.py)
import numpy as np

def awr_update_actor(key, actor, critic, value, batch, temperature):
    v = value(batch.observations)
    critic_output = critic(batch.observations, batch.actions)
    q = _min_q(critic_output)
    exp_a = jnp.exp((q - v) * temperature)
    exp_a = jnp.minimum(exp_a, 100.0)

    def actor_loss_fn(actor_params):
        dist = actor.apply({'params': actor_params},
                           batch.observations,
                           training=True,
                           rngs={'dropout': key})
        log_probs = dist.log_prob(batch.actions)
        actor_loss = -(exp_a * log_probs).mean()
        return actor_loss, {'actor_loss': actor_loss, 'adv': q - v}

    new_actor, info = actor.apply_gradient(actor_loss_fn)
    return new_actor, info

print(" Actor update function defined")

In [ ]:
# Cell 9: IQL Learner (replaces iql/learner.py)
import numpy as np

def target_update(critic, target_critic, tau):
    new_target_params = _tree_map(
        lambda p, tp: p * tau + tp * (1 - tau),
        critic.params, target_critic.params)
    return target_critic.replace(params=new_target_params)


@jax.jit
def _update_jit(rng, actor, critic, value, target_critic, batch,
                discount, tau, expectile, temperature):
    new_value, value_info = update_v(target_critic, value, batch, expectile)
    key, rng = jax.random.split(rng)
    new_actor, actor_info = awr_update_actor(key, actor, target_critic,
                                             new_value, batch, temperature)
    new_critic, critic_info = update_q(critic, new_value, batch, discount)
    new_target_critic = target_update(new_critic, target_critic, tau)

    return rng, new_actor, new_critic, new_value, new_target_critic, {
        **critic_info, **value_info, **actor_info
    }


class Learner(object):
    def __init__(self, seed, observations, actions,
                 actor_lr=3e-4, value_lr=3e-4, critic_lr=3e-4,
                 hidden_dims=(256, 256), discount=0.99, tau=0.005,
                 expectile=0.8, temperature=0.1,
                 dropout_rate=None, max_steps=None,
                 opt_decay_schedule="cosine", num_critics=2):

        self.expectile = expectile
        self.tau = tau
        self.discount = discount
        self.temperature = temperature

        rng = jax.random.PRNGKey(seed)
        rng, actor_key, critic_key, value_key = jax.random.split(rng, 4)

        action_dim = actions.shape[-1]
        actor_def = NormalTanhPolicy(hidden_dims, action_dim,
                                    log_std_scale=1e-3, log_std_min=-5.0,
                                    dropout_rate=dropout_rate,
                                    state_dependent_std=False,
                                    tanh_squash_distribution=False)

        if opt_decay_schedule == "cosine":
            schedule_fn = optax.cosine_decay_schedule(-actor_lr, max_steps)
            optimiser = optax.chain(optax.scale_by_adam(),
                                    optax.scale_by_schedule(schedule_fn))
        else:
            optimiser = optax.adam(learning_rate=actor_lr)

        actor = Model.create(actor_def,
                             inputs=[actor_key, observations],
                             tx=optimiser)

        if num_critics == 3:
            critic_def = TripleCritic(hidden_dims)
        else:
            critic_def = DoubleCritic(hidden_dims)

        critic = Model.create(critic_def,
                              inputs=[critic_key, observations, actions],
                              tx=optax.adam(learning_rate=critic_lr))

        value_def = ValueCritic(hidden_dims)
        value = Model.create(value_def,
                             inputs=[value_key, observations],
                             tx=optax.adam(learning_rate=value_lr))

        target_critic = Model.create(
            critic_def, inputs=[critic_key, observations, actions])

        self.actor = actor
        self.critic = critic
        self.value = value
        self.target_critic = target_critic
        self.rng = rng

    def sample_actions(self, observations, temperature=1.0):
        rng, actions = sample_actions(self.rng, self.actor.apply_fn,
                                     self.actor.params, observations,
                                     temperature)
        self.rng = rng
        actions = np.asarray(actions)
        return np.clip(actions, -1, 1)

    def update(self, batch):
        new_rng, new_actor, new_critic, new_value, new_target_critic, info = _update_jit(
            self.rng, self.actor, self.critic, self.value, self.target_critic,
            batch, self.discount, self.tau, self.expectile, self.temperature)

        self.rng = new_rng
        self.actor = new_actor
        self.critic = new_critic
        self.value = new_value
        self.target_critic = new_target_critic
        return info

print(" IQL Learner defined")

In [ ]:
# Cell 10: Dataset Utilities (replaces iql/dataset_utils.py)
import os
import numpy as np

from tqdm import tqdm

# D4RL env names -> gymnasium MuJoCo env names
_D4RL_TO_GYMNASIUM = {
    'hopper': 'Hopper-v4',
    'halfcheetah': 'HalfCheetah-v4',
    'walker2d': 'Walker2d-v4',
    'ant': 'Ant-v4',
}

def d4rl_to_gymnasium_name(d4rl_name):
    base = d4rl_name.split('-')[0]
    return _D4RL_TO_GYMNASIUM.get(base, d4rl_name)


def split_into_trajectories(observations, actions, rewards, masks,
                            dones_float, next_observations):
    trajs = [[]]
    for i in tqdm(range(len(observations)), desc='Splitting trajectories'):
        trajs[-1].append((observations[i], actions[i], rewards[i], masks[i],
                          dones_float[i], next_observations[i]))
        if dones_float[i] == 1.0 and i + 1 < len(observations):
            trajs.append([])
    return trajs


class Dataset(object):
    def __init__(self, observations, actions, rewards, masks,
                 dones_float, next_observations, size):
        self.observations = observations
        self.actions = actions
        self.rewards = rewards
        self.masks = masks
        self.dones_float = dones_float
        self.next_observations = next_observations
        self.size = size

    def sample(self, batch_size):
        indx = np.random.randint(self.size, size=batch_size)
        return Batch(observations=self.observations[indx],
                     actions=self.actions[indx],
                     rewards=self.rewards[indx],
                     masks=self.masks[indx],
                     next_observations=self.next_observations[indx])


def _d4rl_url_filename(env_name):
    parts = env_name.split('-')
    if len(parts) >= 3 and parts[-1].startswith('v'):
        env_part = parts[0]
        version = parts[-1]
        dataset_parts = parts[1:-1]
        return f"{env_part}_{'_'.join(dataset_parts)}-{version}.hdf5"
    return env_name.replace('-', '_', 1) + '.hdf5'


def _load_from_hdf5(cache_path):
    import h5py
    with h5py.File(cache_path, 'r') as f:
        dataset = {
            'observations': f['observations'][:],
            'actions': f['actions'][:],
            'rewards': f['rewards'][:],
            'terminals': f['terminals'][:],
            'next_observations': np.concatenate([
                f['observations'][1:],
                f['observations'][-1:],
            ], axis=0),
        }
    return dataset


def _load_d4rl_dataset(env_or_name):
    name = env_or_name if isinstance(env_or_name, str) else getattr(
        getattr(env_or_name, 'spec', None), 'id', str(env_or_name))
    url_filename = _d4rl_url_filename(name)
    cache_dir = os.path.join(os.path.expanduser('~'), '.d4rl', 'datasets')
    cache_path = os.path.join(cache_dir, url_filename)

    if os.path.exists(cache_path):
        print(f"Loading cached dataset: {cache_path}")
        return _load_from_hdf5(cache_path)

    # Download the HDF5 file directly from D4RL URLs
    url = f"http://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/{url_filename}"
    os.makedirs(cache_dir, exist_ok=True)
    print(f"Downloading D4RL dataset: {url} ...")
    urllib.request.urlretrieve(url, cache_path)
    print(f"Saved to {cache_path}")
    return _load_from_hdf5(cache_path)


class D4RLDataset(Dataset):
    def __init__(self, env_or_name, clip_to_eps=True, eps=1e-5):
        dataset = _load_d4rl_dataset(env_or_name)
        if clip_to_eps:
            lim = 1 - eps
            dataset['actions'] = np.clip(dataset['actions'], -lim, lim)

        dones_float = np.zeros_like(dataset['rewards'])
        for i in range(len(dones_float) - 1):
            if np.linalg.norm(dataset['observations'][i + 1] -
                              dataset['next_observations'][i]
                              ) > 1e-6 or dataset['terminals'][i] == 1.0:
                dones_float[i] = 1
            else:
                dones_float[i] = 0
        dones_float[-1] = 1

        super().__init__(
            dataset['observations'].astype(np.float32),
            actions=dataset['actions'].astype(np.float32),
            rewards=dataset['rewards'].astype(np.float32),
            masks=1.0 - dataset['terminals'].astype(np.float32),
            dones_float=dones_float.astype(np.float32),
            next_observations=dataset['next_observations'].astype(np.float32),
            size=len(dataset['observations']))

print(" Dataset utilities defined")

In [ ]:
# Cell 11: Wrappers (replaces wrappers/ package)

# --- EpisodeMonitor ---
import time
import copy
import numpy as np

class EpisodeMonitor(gym.ActionWrapper):
    """Computes episode returns and lengths."""
    def __init__(self, env):
        super().__init__(env)
        self._reset_stats()
        self.total_timesteps = 0

    def _reset_stats(self):
        self.reward_sum = 0.0
        self.episode_length = 0
        self.start_time = time.time()

    def step(self, action):
        step_result = self.env.step(action)
        if len(step_result) == 5:
            observation, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            observation, reward, done, info = step_result
            terminated = done
            truncated = False

        self.reward_sum += reward
        self.episode_length += 1
        self.total_timesteps += 1
        info['total'] = {'timesteps': self.total_timesteps}

        if done:
            info['episode'] = {
                'return': self.reward_sum,
                'length': self.episode_length,
                'duration': time.time() - self.start_time,
            }
        return observation, reward, terminated, truncated, info

    def reset(self, **kwargs):
        self._reset_stats()
        return self.env.reset(**kwargs)


# --- SinglePrecision ---
class SinglePrecision(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        if isinstance(self.observation_space, Box):
            obs_space = self.observation_space
            self.observation_space = Box(obs_space.low, obs_space.high,
                                         obs_space.shape)
        elif isinstance(self.observation_space, GymDict):
            obs_spaces = copy.copy(self.observation_space.spaces)
            for k, v in obs_spaces.items():
                obs_spaces[k] = Box(v.low, v.high, v.shape)
            self.observation_space = GymDict(obs_spaces)

    def observation(self, observation):
        if isinstance(observation, np.ndarray):
            return observation.astype(np.float32)
        elif isinstance(observation, dict):
            observation = copy.copy(observation)
            for k, v in observation.items():
                observation[k] = v.astype(np.float32)
            return observation


# --- GravityShift ---
class GravityShift(gym.Wrapper):
    def __init__(self, env, gravity_scale=1.0):
        super().__init__(env)
        self.gravity_scale = gravity_scale
        self._default_gravity = None

    def reset(self, **kwargs):
        obs = self.env.reset(**kwargs)
        if self._default_gravity is None:
            self._default_gravity = self.env.unwrapped.model.opt.gravity.copy()
        self.env.unwrapped.model.opt.gravity[:] = (
            self._default_gravity * self.gravity_scale)
        return obs


# --- ObservationNoise ---
class ObservationNoise(gym.ObservationWrapper):
    def __init__(self, env, noise_std=0.0):
        super().__init__(env)
        self.noise_std = noise_std

    def observation(self, observation):
        if self.noise_std > 0:
            noise = np.random.normal(
                0, self.noise_std, size=observation.shape
            ).astype(observation.dtype)
            return observation + noise
        return observation


# --- FrictionShift ---
class FrictionShift(gym.Wrapper):
    def __init__(self, env, friction_scale=1.0):
        super().__init__(env)
        self.friction_scale = friction_scale
        self._default_friction = None

    def reset(self, **kwargs):
        obs = self.env.reset(**kwargs)
        if self._default_friction is None:
            self._default_friction = self.env.unwrapped.model.geom_friction.copy()
        self.env.unwrapped.model.geom_friction[:] = (
            self._default_friction * self.friction_scale)
        return obs


# --- RewardPerturbation ---
class RewardPerturbation(gym.Wrapper):
    def __init__(self, env, noise_std=0.0, scale=1.0):
        super().__init__(env)
        self.noise_std = noise_std
        self.scale = scale

    def step(self, action):
        step_result = self.env.step(action)
        if len(step_result) == 5:
            observation, reward, terminated, truncated, info = step_result
        else:
            observation, reward, done, info = step_result
            terminated = done
            truncated = False
        perturbed_reward = self.scale * reward
        if self.noise_std > 0:
            perturbed_reward += np.random.normal(0, self.noise_std)
        return observation, perturbed_reward, terminated, truncated, info

print(" All wrappers defined")

In [ ]:
# Cell 12: Evaluation Function (replaces evaluation/evaluate.py)
import numpy as np

def evaluate(agent, env, num_episodes):
    stats = {'return': [], 'length': []}
    for _ in range(num_episodes):
        observation = env.reset()
        if isinstance(observation, tuple):
            observation = observation[0]
        done = False
        while not done:
            action = agent.sample_actions(observation, temperature=0.0)
            step_result = env.step(action)
            if len(step_result) == 5:
                observation, _, terminated, truncated, info = step_result
                done = terminated or truncated
            else:
                observation, _, done, info = step_result
        for k in stats.keys():
            stats[k].append(info['episode'][k])
    for k, v in stats.items():
        stats[k] = np.mean(v)
    return stats

print(" Evaluation function defined")

## What is IQL?

**Implicit Q-Learning** (Kostrikov et al., ICLR 2022) learns Q-values from a fixed offline dataset — no environment interaction during training.

**Key idea:** Use *expectile regression* to approximate the max over actions without querying out-of-distribution actions:

$$L_\tau^{V}(\psi) = \mathbb{E}_{(s,a)\sim\mathcal{D}}\left[|\tau - \mathbf{1}(Q_{\hat{\theta}}(s,a) - V_\psi(s) < 0)| \cdot (Q_{\hat{\theta}}(s,a) - V_\psi(s))^2\right]$$

The actor is updated via **advantage-weighted regression**: $\pi(a|s) \propto \exp\left(\beta \cdot (Q(s,a) - V(s))\right)$

**Our extension:** Replace `min(Q₁, Q₂)` with `min(Q₁, Q₂, Q₃)` for more conservative value estimates.

In [ ]:
# Cell 14: Configuration
ENV_NAMES = [
    'hopper-medium-v2',
    'halfcheetah-medium-v2',
    'walker2d-medium-v2',
]
DEMO_STEPS = 20_000        # Quick demo (vs 300k for production) — reduced for Colab T4 memory
EVAL_INTERVAL = 10_000     # Evaluate every 10k steps
EVAL_EPISODES = 5          # Reduced from 10 to save memory on Colab T4
SEED = 42
BATCH_SIZE = 256

CONFIG = {
    'actor_lr': 3e-4, 'value_lr': 3e-4, 'critic_lr': 3e-4,
    'hidden_dims': (256, 256), 'discount': 0.99,
    'expectile': 0.7, 'temperature': 3.0,
    'dropout_rate': None, 'tau': 0.005,
}

# Shift configurations — 4 types of distribution shift
SHIFT_CONFIG = {
    'gravity':        [0.5, 1.0, 1.5, 2.0],
    'obs_noise':      [0.0, 0.01, 0.1, 0.3],
    'friction':       [0.5, 1.0, 1.5, 2.0],
    'reward_perturb': [0.0, 0.1, 0.5, 1.0],
}

# Baseline (no-shift) levels for computing robustness drop
BASELINE_LEVELS = {
    'gravity': 1.0, 'obs_noise': 0.0,
    'friction': 1.0, 'reward_perturb': 0.0,
}

# Accumulators for cross-environment results
ALL_RESULTS = {}   # {env_name: {'results_2q': [...], 'results_3q': [...], ...}}

print(f"Environments: {ENV_NAMES}")
print(f"Training steps: {DEMO_STEPS:,}")
print(f"Shift types: {list(SHIFT_CONFIG.keys())}")

## Steps 1–4: Train, Evaluate, and Plot for All Environments

We loop over **3 MuJoCo environments** (Hopper, HalfCheetah, Walker2d).
For each environment we:
1. Load the D4RL offline dataset
2. Train 2Q (DoubleCritic) and 3Q (TripleCritic) IQL agents
3. Evaluate under 4 types of distribution shift
4. Save plots and agent params for video recording

Training is **purely offline** — we only sample mini-batches from the fixed dataset.

In [ ]:
# Helper functions (used inside the loop)

def normalize(dataset):
    """Normalize rewards by trajectory return range (x1000)."""
    trajs = split_into_trajectories(
        dataset.observations, dataset.actions, dataset.rewards,
        dataset.masks, dataset.dones_float, dataset.next_observations)
    def compute_returns(traj):
        return sum(rew for _, _, rew, _, _, _ in traj)
    trajs.sort(key=compute_returns)
    ret_range = compute_returns(trajs[-1]) - compute_returns(trajs[0])
    if abs(ret_range) < 1e-8:
        print(f'WARNING: reward range is ~0, skipping normalization')
    else:
        dataset.rewards /= ret_range
        dataset.rewards *= 1000.0

def make_env(env_name, seed):
    """Create a gymnasium environment with monitoring wrappers."""
    gym_env_name = d4rl_to_gymnasium_name(env_name)
    env = gym.make(gym_env_name)
    env = EpisodeMonitor(env)
    env = SinglePrecision(env)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return env

def train_iql(env_name, dataset, num_critics, max_steps, eval_interval, seed, config):
    """Train IQL and return the agent + eval history."""
    env = make_env(env_name, seed)
    label = f"{num_critics}Q"

    agent = Learner(
        seed, env.observation_space.sample()[np.newaxis],
        env.action_space.sample()[np.newaxis],
        max_steps=max_steps, num_critics=num_critics, **config)

    eval_history = []
    start_time = time.time()

    print(f"\n{'='*50}")
    print(f"Training IQL ({label}) on {env_name}")
    print(f"Steps: {max_steps:,} | Eval every: {eval_interval:,}")
    print(f"{'='*50}")

    for i in tqdm(range(1, max_steps + 1), desc=f'{label}'):
        batch = dataset.sample(BATCH_SIZE)
        info = agent.update(batch)

        if i % eval_interval == 0:
            stats = evaluate(agent, env, EVAL_EPISODES)
            elapsed = time.time() - start_time
            eval_history.append({
                'step': i, 'return': stats['return'],
                'length': stats['length'], 'time': elapsed
            })
            print(f"  Step {i:>6,} | Return: {stats['return']:>8.2f} | "
                  f"Length: {stats['length']:>6.0f} | Time: {elapsed:.1f}s")

    total_time = time.time() - start_time
    print(f"\n {label} training complete in {total_time:.1f}s")
    env.close()
    return agent, eval_history

def evaluate_shifts(agent, env_name, shift_config, baseline_levels, eval_episodes, seed):
    """Evaluate agent under all shift types and levels."""
    all_results = []

    for shift_type, levels in shift_config.items():
        print(f"\n   {shift_type}:")
        for level in levels:
            gym_env_name = d4rl_to_gymnasium_name(env_name)
            env = gym.make(gym_env_name)

            if shift_type == 'gravity':
                env = GravityShift(env, gravity_scale=level)
            elif shift_type == 'obs_noise':
                env = ObservationNoise(env, noise_std=level)
            elif shift_type == 'friction':
                env = FrictionShift(env, friction_scale=level)
            elif shift_type == 'reward_perturb':
                env = RewardPerturbation(env, noise_std=level)

            env = EpisodeMonitor(env)
            env = SinglePrecision(env)
            env.reset(seed=seed)
            env.action_space.seed(seed)

            stats = evaluate(agent, env, eval_episodes)
            all_results.append({
                'shift_type': shift_type, 'shift_level': level,
                'raw_return': stats['return'], 'episode_length': stats['length'],
            })
            print(f"    {shift_type}={level:.2f}: return={stats['return']:.2f}")
            env.close()

    # Compute robustness drop relative to baseline level
    baseline_scores = {}
    for r in all_results:
        bl = baseline_levels.get(r['shift_type'], 0)
        if r['shift_level'] == bl:
            baseline_scores[r['shift_type']] = r['raw_return']

    for r in all_results:
        baseline = baseline_scores.get(r['shift_type'], r['raw_return'])
        if baseline == 0:
            r['robustness_drop'] = 0.0
        else:
            r['robustness_drop'] = (baseline - r['raw_return']) / abs(baseline)

    return all_results

print('Helper functions defined.')

In [ ]:
# Main pipeline: Train + Evaluate + Plot for ALL environments
import time, gc, pickle, os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('demo_outputs/plots', exist_ok=True)
os.makedirs('demo_outputs/videos', exist_ok=True)

shift_types = ['gravity', 'obs_noise', 'friction', 'reward_perturb']
shift_labels = {
    'gravity': ('Gravity Scale', 'Gravity Shift'),
    'obs_noise': ('Noise σ', 'Observation Noise'),
    'friction': ('Friction Scale', 'Friction Shift'),
    'reward_perturb': ('Reward Noise σ', 'Reward Perturbation'),
}

for env_idx, ENV_NAME in enumerate(ENV_NAMES):
    env_short = ENV_NAME.split('-')[0]  # 'hopper', 'halfcheetah', 'walker2d'
    print(f'\n{"#"*60}')
    print(f'# Environment {env_idx+1}/{len(ENV_NAMES)}: {ENV_NAME}')
    print(f'{"#"*60}')

    # ── 1. Load dataset ──
    print(f'\n📦 Loading D4RL dataset: {ENV_NAME}...')
    dataset = D4RLDataset(ENV_NAME)
    normalize(dataset)
    print(f'   Dataset size: {dataset.size:,} transitions')
    print(f'   Observation shape: {dataset.observations.shape}')
    print(f'   Action shape: {dataset.actions.shape}')

    # ── 2. Train 2Q ──
    agent_2q, history_2q = train_iql(
        ENV_NAME, dataset, num_critics=2,
        max_steps=DEMO_STEPS, eval_interval=EVAL_INTERVAL,
        seed=SEED, config=CONFIG)

    gc.collect()
    if hasattr(jax, 'clear_caches'):
        jax.clear_caches()

    # ── 3. Train 3Q ──
    agent_3q, history_3q = train_iql(
        ENV_NAME, dataset, num_critics=3,
        max_steps=DEMO_STEPS, eval_interval=EVAL_INTERVAL,
        seed=SEED, config=CONFIG)

    # ── 4. Evaluate under distribution shift ──
    print(f'\n📊 Evaluating 2Q under distribution shift...')
    results_2q = evaluate_shifts(agent_2q, ENV_NAME, SHIFT_CONFIG, BASELINE_LEVELS, EVAL_EPISODES, SEED)
    print(f'\n📊 Evaluating 3Q under distribution shift...')
    results_3q = evaluate_shifts(agent_3q, ENV_NAME, SHIFT_CONFIG, BASELINE_LEVELS, EVAL_EPISODES, SEED)

    # ── 5. Plot training curves ──
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot([h['step'] for h in history_2q], [h['return'] for h in history_2q],
            '-o', label='Baseline (2Q)', color='steelblue', linewidth=2, markersize=8)
    ax.plot([h['step'] for h in history_3q], [h['return'] for h in history_3q],
            '--s', label='Q-Ensemble (3Q)', color='coral', linewidth=2, markersize=8)
    ax.set_xlabel('Training Steps', fontsize=12)
    ax.set_ylabel('Average Return', fontsize=12)
    ax.set_title(f'Training Curves — {ENV_NAME} (Demo: {DEMO_STEPS:,} steps)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'demo_outputs/plots/training_curves_{env_short}.png', dpi=150, bbox_inches='tight')
    print(f'  📈 Saved: demo_outputs/plots/training_curves_{env_short}.png')
    plt.show()

    # ── 6. Plot degradation curves ──
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, shift_type in enumerate(shift_types):
        ax = axes[idx // 2, idx % 2]
        xlabel, title = shift_labels[shift_type]
        data_2q = sorted([r for r in results_2q if r['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        data_3q = sorted([r for r in results_3q if r['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        ax.plot([r['shift_level'] for r in data_2q], [r['raw_return'] for r in data_2q],
                '-o', label='Baseline (2Q)', color='steelblue', linewidth=2, markersize=8)
        ax.plot([r['shift_level'] for r in data_3q], [r['raw_return'] for r in data_3q],
                '--s', label='Q-Ensemble (3Q)', color='coral', linewidth=2, markersize=8)
        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel('Average Return', fontsize=11)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'Performance Under Distribution Shift — {ENV_NAME}\n(Demo: {DEMO_STEPS:,} training steps)',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'demo_outputs/plots/degradation_curves_{env_short}.png', dpi=150, bbox_inches='tight')
    print(f'  📈 Saved: demo_outputs/plots/degradation_curves_{env_short}.png')
    plt.show()

    # ── 7. Plot robustness drop bars ──
    fig, axes_bar = plt.subplots(1, 4, figsize=(20, 5))
    for idx, shift_type in enumerate(shift_types):
        ax = axes_bar[idx]
        _, title = shift_labels[shift_type]
        data_2q = sorted([r for r in results_2q if r['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        data_3q = sorted([r for r in results_3q if r['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        levels = [r['shift_level'] for r in data_2q]
        drops_2q = [r['robustness_drop'] for r in data_2q]
        drops_3q = [r['robustness_drop'] for r in data_3q]
        x = np.arange(len(levels))
        w = 0.35
        ax.bar(x - w/2, drops_2q, w, label='2Q', color='steelblue', alpha=0.8)
        ax.bar(x + w/2, drops_3q, w, label='3Q', color='coral', alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels([f'{l:.2f}' for l in levels])
        ax.set_ylabel('Δ(δ) Robustness Drop')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
    plt.suptitle(f'Robustness Drop — {ENV_NAME}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'demo_outputs/plots/robustness_drop_bars_{env_short}.png', dpi=150, bbox_inches='tight')
    print(f'  📈 Saved: demo_outputs/plots/robustness_drop_bars_{env_short}.png')
    plt.show()

    # ── 8. Save agent params for video recording ──
    params_path = f'demo_outputs/_agent_2q_params_{env_short}.pkl'
    with open(params_path, 'wb') as f:
        pickle.dump({
            'actor_params': jax.device_get(agent_2q.actor.params),
            'env_name': ENV_NAME,
            'hidden_dims': (256, 256),
        }, f)
    print(f'  💾 Saved agent params: {params_path}')

    # ── 9. Save CSVs ──
    import csv
    for label, results in [('2Q', results_2q), ('3Q', results_3q)]:
        outfile = f'demo_outputs/demo_shift_{ENV_NAME}_{label}.csv'
        with open(outfile, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=[
                'shift_type', 'shift_level', 'raw_return', 'episode_length', 'robustness_drop'])
            writer.writeheader()
            writer.writerows(results)
    for label, history in [('2Q', history_2q), ('3Q', history_3q)]:
        outfile = f'demo_outputs/demo_training_{ENV_NAME}_{label}.csv'
        with open(outfile, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['step', 'return', 'length', 'time'])
            writer.writeheader()
            writer.writerows(history)
    print(f'  💾 Saved CSVs for {ENV_NAME}')

    # ── 10. Store results and free memory ──
    ALL_RESULTS[ENV_NAME] = {
        'results_2q': results_2q, 'results_3q': results_3q,
        'history_2q': history_2q, 'history_3q': history_3q,
    }

    del agent_2q, agent_3q, dataset
    gc.collect()
    if hasattr(jax, 'clear_caches'):
        jax.clear_caches()
    gc.collect()
    print(f'  🧹 Memory cleanup done for {ENV_NAME}')

# Save checkpoint with ALL environments
checkpoint_path = 'demo_outputs/_checkpoint.pkl'
with open(checkpoint_path, 'wb') as f:
    pickle.dump({
        'ALL_RESULTS': ALL_RESULTS,
        'ENV_NAMES': ENV_NAMES,
        'SEED': SEED,
        'DEMO_STEPS': DEMO_STEPS,
    }, f)
print(f'\n✅ All environments complete! Checkpoint saved to {checkpoint_path}')
print('If the kernel crashes during video recording, use the Restart Recovery cell.')

### ⚡ Restart Recovery (run this cell ONLY after a kernel crash)

If the kernel crashed during video recording, run **only this cell** to restore state, then skip to the video recording cells below. No need to re-train!

In [ ]:
# ⚡ RESTART RECOVERY — Run this cell after a kernel crash to skip training.
# It restores all saved state and jumps straight to video recording.
# SKIP this cell if you're running the notebook from scratch.

import os, sys, pickle
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.4'

checkpoint_path = 'demo_outputs/_checkpoint.pkl'

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        ckpt = pickle.load(f)
    ALL_RESULTS = ckpt['ALL_RESULTS']
    ENV_NAMES = ckpt['ENV_NAMES']
    SEED = ckpt['SEED']
    DEMO_STEPS = ckpt['DEMO_STEPS']
    IN_COLAB = 'google.colab' in sys.modules
    print(f'✅ Restored checkpoint: {len(ALL_RESULTS)} environments, seed={SEED}, {DEMO_STEPS:,} steps')
    for env_name in ALL_RESULTS:
        env_short = env_name.split('-')[0]
        r = ALL_RESULTS[env_name]
        print(f'   {env_name}: {len(r["results_2q"])} 2Q evals, {len(r["results_3q"])} 3Q evals')
        params_path = f'demo_outputs/_agent_2q_params_{env_short}.pkl'
        print(f'     Agent params: {params_path} ({"✓" if os.path.exists(params_path) else "✗ MISSING"})')
    print(f'\n→ Now run the video recording cells below (skip all training cells).')
else:
    print('No checkpoint found — run the full notebook from the top.')
    print(f'  Looked for: {checkpoint_path}')

In [ ]:
# Cell 27c: Memory diagnostics — see exactly what's using GPU/CPU RAM
import subprocess, os

print('=== Memory Diagnostics ===')

# GPU memory (nvidia-smi)
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total',
                        '--format=csv,noheader,nounits'], capture_output=True, text=True)
    if r.returncode == 0:
        used, free, total = [int(x.strip()) for x in r.stdout.strip().split(',')]
        print(f'GPU Memory: {used} MB used / {free} MB free / {total} MB total')
        print(f'GPU Memory usage: {used/total*100:.1f}%')
    else:
        print('nvidia-smi not available (CPU-only runtime?)')
except FileNotFoundError:
    print('nvidia-smi not found (CPU-only runtime)')

# CPU/System memory
try:
    with open('/proc/meminfo') as f:
        meminfo = {}
        for line in f:
            parts = line.split(':')
            if len(parts) == 2:
                meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
    total_mb = meminfo.get('MemTotal', 0) // 1024
    avail_mb = meminfo.get('MemAvailable', 0) // 1024
    used_mb = total_mb - avail_mb
    print(f'System RAM: {used_mb} MB used / {avail_mb} MB free / {total_mb} MB total')
    print(f'System RAM usage: {used_mb/total_mb*100:.1f}%')
except Exception:
    print('Could not read /proc/meminfo (not Linux?)')

# Python process memory
try:
    import resource
    rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # On macOS ru_maxrss is in bytes, on Linux it's in KB
    import platform
    if platform.system() == 'Darwin':
        rss_mb = rss_kb / (1024 * 1024)
    else:
        rss_mb = rss_kb / 1024
    print(f'Python process peak RSS: {rss_mb:.0f} MB')
except Exception:
    pass

print(f'MUJOCO_GL = {os.environ.get("MUJOCO_GL", "not set")}')
print('===========================')

##  Step 5: Record Agent Videos

We record MuJoCo videos of the **2Q agent** under 4 conditions (baseline + 3 shift types).
This visually demonstrates how distribution shift degrades agent behavior.

**Strategy:** Videos are recorded in a **subprocess** to isolate MuJoCo rendering from JAX's GPU memory. If the subprocess gets OOM-killed, the notebook kernel survives and we skip videos gracefully.

In [ ]:
# Write a standalone video recording script to disk.
# This script runs in a SEPARATE process that never imports JAX,
# so it cannot conflict with JAX's GPU memory allocation.
# It uses osmesa (CPU) rendering — no GPU memory needed at all.

video_script = r'''
#!/usr/bin/env python3
"""Standalone MuJoCo video recorder — runs WITHOUT JAX."""
import os, sys, pickle, json
os.environ['MUJOCO_GL'] = 'osmesa'  # Force CPU rendering
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'  # Don't let JAX grab GPU if accidentally imported

import numpy as np
import gymnasium as gym
import imageio

# ── Minimal wrappers (no JAX dependency) ──
class GravityShift(gym.Wrapper):
    def __init__(self, env, gravity_scale=1.0):
        super().__init__(env)
        self.gravity_scale = gravity_scale
        model = env.unwrapped.model
        model.opt.gravity[2] = -9.81 * gravity_scale

class ObservationNoise(gym.ObservationWrapper):
    def __init__(self, env, noise_std=0.1):
        super().__init__(env)
        self.noise_std = noise_std
        self._rng = np.random.RandomState(0)
    def observation(self, obs):
        return obs + self._rng.normal(0, self.noise_std, size=obs.shape).astype(obs.dtype)

class FrictionShift(gym.Wrapper):
    def __init__(self, env, friction_scale=1.0):
        super().__init__(env)
        model = env.unwrapped.model
        model.geom_friction[:] *= friction_scale

# ── Minimal actor (numpy-only, no JAX) ──
def _to_numpy(x):
    """Convert any array-like (JAX, numpy, dict) to plain numpy."""
    if isinstance(x, dict):
        return {k: _to_numpy(v) for k, v in x.items()}
    return np.array(x, dtype=np.float32)

def sample_action_numpy(actor_params, obs):
    """Deterministic action from NormalTanhPolicy using numpy only.
    
    Flax NormalTanhPolicy_0 param structure:
      NormalTanhPolicy_0/
        MLP_0/
          Dense_0/ {kernel, bias}  # hidden layer 1 (256)
          Dense_1/ {kernel, bias}  # hidden layer 2 (256)
        Dense_0/ {kernel, bias}    # mean head (action_dim)
        Dense_1/ {kernel, bias}    # log_std head (action_dim)
    """
    x = np.array(obs, dtype=np.float32).reshape(1, -1)
    
    # Navigate to policy params
    p = actor_params
    for key in ["NormalTanhPolicy_0", "TanhPolicy_0"]:
        if key in p:
            p = p[key]
            break
    
    # Forward through MLP backbone (with relu activations)
    if "MLP_0" in p:
        mlp = p["MLP_0"]
        for layer_name in sorted(mlp.keys()):  # Dense_0, Dense_1, ...
            w = np.array(mlp[layer_name]["kernel"], dtype=np.float32)
            b = np.array(mlp[layer_name]["bias"], dtype=np.float32)
            x = x @ w + b
            x = np.maximum(x, 0)  # relu (all MLP layers have activation)
    
    # Mean head: Dense_0 at policy level (NOT inside MLP_0)
    if "Dense_0" in p:
        w = np.array(p["Dense_0"]["kernel"], dtype=np.float32)
        b = np.array(p["Dense_0"]["bias"], dtype=np.float32)
        mean = x @ w + b
    else:
        mean = x  # fallback
    
    return np.tanh(mean).flatten()  # tanh squashing


D4RL_TO_GYM = {
    "hopper-medium-v2": "Hopper-v4",
    "halfcheetah-medium-v2": "HalfCheetah-v4",
    "walker2d-medium-v2": "Walker2d-v4",
    "hopper-medium-expert-v2": "Hopper-v4",
    "halfcheetah-medium-expert-v2": "HalfCheetah-v4",
    "walker2d-medium-expert-v2": "Walker2d-v4",
}

def record_video(actor_params, env_name, filename, shift_type=None, shift_level=None,
                 max_steps=200, fps=30):
    gym_name = D4RL_TO_GYM.get(env_name, env_name)
    print(f"    Creating {gym_name} with render_mode=rgb_array...")
    env = gym.make(gym_name, render_mode="rgb_array")
    print(f"    Environment created (MUJOCO_GL={os.environ.get('MUJOCO_GL', 'default')})")

    if shift_type == "gravity":
        env = GravityShift(env, gravity_scale=shift_level)
    elif shift_type == "obs_noise":
        env = ObservationNoise(env, noise_std=shift_level)
    elif shift_type == "friction":
        env = FrictionShift(env, friction_scale=shift_level)

    print(f"    Resetting environment...")
    obs, _ = env.reset(seed=42)
    frames = []
    total_reward = 0.0

    print(f"    Running episode (max {max_steps} steps)...")
    for step in range(max_steps):
        frame = env.render()
        frames.append(frame)
        action = sample_action_numpy(actor_params, obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    print(f"    Episode done: {len(frames)} frames, return={total_reward:.2f}")
    env.close()

    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)
    imageio.mimsave(filename, frames, fps=fps)
    print(f"    Saved: {filename}")
    return total_reward


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--params", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--shift_type", default=None)
    parser.add_argument("--shift_level", type=float, default=None)
    parser.add_argument("--max_steps", type=int, default=200)
    args = parser.parse_args()

    with open(args.params, "rb") as f:
        data = pickle.load(f)

    ret = record_video(
        data["actor_params"], data["env_name"], args.output,
        shift_type=args.shift_type, shift_level=args.shift_level,
        max_steps=args.max_steps,
    )
    print(json.dumps({"return": ret, "status": "ok"}))
'''

script_path = 'demo_outputs/_record_video.py'
with open(script_path, 'w') as f:
    f.write(video_script)
print(f'Wrote video recording script to {script_path}')

In [ ]:
# Record videos in SUBPROCESSES for ALL environments.
# Isolates MuJoCo from JAX GPU memory. If a subprocess gets OOM-killed,
# the notebook kernel survives.
import subprocess, os, sys, json

os.makedirs('demo_outputs/videos', exist_ok=True)
script_path = 'demo_outputs/_record_video.py'
total_videos = 0
total_recorded = 0

SHIFT_CONFIGS = [
    ('Baseline (no shift)', 'baseline', None, None),
    ('Gravity shift (2x)', 'gravity2x', 'gravity', 2.0),
    ('Observation noise (σ=0.3)', 'noise0.3', 'obs_noise', 0.3),
    ('Friction shift (0.5x)', 'friction0.5', 'friction', 0.5),
]

for ENV_NAME in ENV_NAMES:
    env_short = ENV_NAME.split('-')[0]
    params_path = f'demo_outputs/_agent_2q_params_{env_short}.pkl'

    if not os.path.exists(params_path):
        print(f'\n⚠️  Skipping {ENV_NAME}: params not found at {params_path}')
        continue

    print(f'\n🎬 Recording 2Q videos for {ENV_NAME} (4 conditions)...')
    for label, suffix, stype, slevel in SHIFT_CONFIGS:
        filepath = f'demo_outputs/videos/{env_short}_2Q_{suffix}.mp4'
        total_videos += 1
        print(f'  {label}:')
        cmd = [
            sys.executable, script_path,
            '--params', params_path,
            '--output', filepath,
            '--max_steps', '200',
        ]
        if stype is not None:
            cmd += ['--shift_type', stype, '--shift_level', str(slevel)]

        try:
            result = subprocess.run(
                cmd, capture_output=True, text=True, timeout=120,
                env={**os.environ, 'MUJOCO_GL': 'osmesa',
                     'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'},
            )
            if result.stdout:
                for line in result.stdout.strip().split('\n'):
                    print(f'    {line}')
            if result.returncode != 0:
                print(f'    ❌ FAILED (exit code {result.returncode})')
                if result.stderr:
                    for line in result.stderr.strip().split('\n')[-3:]:
                        print(f'    stderr: {line}')
                if result.returncode == -9:
                    print('    (Process was OOM-killed by the OS)')
            else:
                total_recorded += 1
        except subprocess.TimeoutExpired:
            print(f'    ❌ TIMEOUT (>120s)')
        except Exception as e:
            print(f'    ❌ ERROR: {type(e).__name__}: {e}')

print(f'\n✅ Video recording complete: {total_recorded}/{total_videos} videos saved.')
if total_recorded == 0:
    print('\n⚠️  No videos were recorded. This is common on Colab due to memory constraints.')
    print('All plots and CSVs were already saved successfully.')

In [ ]:
# Display videos inline (Colab) or list files (local)
import os, glob

SHIFT_SUFFIXES = {
    'baseline': 'Baseline (No Shift)',
    'gravity2x': '2x Gravity',
    'noise0.3': 'Observation Noise (σ=0.3)',
    'friction0.5': '0.5x Friction (Slippery)',
}

all_videos = sorted(glob.glob('demo_outputs/videos/*_2Q_*.mp4'))

if not all_videos:
    print('No videos were recorded (see errors above). Skipping display.')
elif IN_COLAB:
    from IPython.display import HTML, display as ipy_display
    from base64 import b64encode

    def show_video(path, title=''):
        mp4 = open(path, 'rb').read()
        b64 = b64encode(mp4).decode()
        return f'''
        <div style="display:inline-block; margin:10px; text-align:center;">
            <h4>{title}</h4>
            <video width="250" controls autoplay loop muted>
                <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
        </div>'''

    for ENV_NAME in ENV_NAMES:
        env_short = ENV_NAME.split('-')[0]
        env_videos = [v for v in all_videos if f'/{env_short}_' in v]
        if env_videos:
            html = f'<h3>{ENV_NAME} — 2Q Agent Under Distribution Shift</h3><div>'
            for vpath in env_videos:
                fname = os.path.basename(vpath)
                suffix = fname.replace(f'{env_short}_2Q_', '').replace('.mp4', '')
                title = SHIFT_SUFFIXES.get(suffix, suffix)
                html += show_video(vpath, title)
            html += '</div>'
            ipy_display(HTML(html))
else:
    print('Videos saved to demo_outputs/videos/')
    for vpath in all_videos:
        size_kb = os.path.getsize(vpath) / 1024
        print(f'  {vpath} ({size_kb:.1f} KB)')

## Step 6: Compare with Full Production Results (300k steps)

We load pre-computed results from the `results/` directory (trained for 300k steps) and overlay them with our quick 50k-step demo results. This shows how additional training affects robustness.

In [ ]:
# Cell 33: Load pre-computed results (300k steps) for all environments
# On Colab, download them from GitHub since the results/ dir isn't available.
import os, csv, urllib.request

def load_csv(filepath):
    """Load a CSV file and auto-convert numeric fields."""
    rows = []
    with open(filepath) as f:
        for row in csv.DictReader(f):
            for k in row:
                try:
                    row[k] = float(row[k])
                except (ValueError, TypeError):
                    pass
            rows.append(row)
    return rows

# Try local first, then download from GitHub
results_dir = 'results'
if not os.path.exists(results_dir):
    results_dir = '../results'

GITHUB_RAW = 'https://raw.githubusercontent.com/shloakk/iql-robustness-analysis/main/results'

# Download pre-computed results from GitHub if not available locally
if not os.path.exists(results_dir):
    results_dir = 'results'
    os.makedirs(results_dir, exist_ok=True)
    print('📥 Downloading pre-computed results from GitHub...')
    for env_name in ENV_NAMES:
        for label in ['2Q', '3Q']:
            fname = f'shift_{env_name}_{label}_seed{SEED}.csv'
            local_path = os.path.join(results_dir, fname)
            if not os.path.exists(local_path):
                url = f'{GITHUB_RAW}/{fname}'
                try:
                    urllib.request.urlretrieve(url, local_path)
                    print(f'   ✅ {fname}')
                except Exception as e:
                    print(f'   ❌ {fname}: {e}')

PRECOMPUTED = {}  # {env_name: {'2q': [...], '3q': [...]}}

for env_name in ENV_NAMES:
    p2q = os.path.join(results_dir, f'shift_{env_name}_2Q_seed{SEED}.csv')
    p3q = os.path.join(results_dir, f'shift_{env_name}_3Q_seed{SEED}.csv')
    if os.path.exists(p2q) and os.path.exists(p3q):
        PRECOMPUTED[env_name] = {'2q': load_csv(p2q), '3q': load_csv(p3q)}
        print(f'✅ Loaded pre-computed results for {env_name}')
    else:
        print(f'⚠️  Pre-computed results not found for {env_name}')

if not PRECOMPUTED:
    print('\nNo pre-computed results available for comparison.')

In [ ]:
# Cell 34: Side-by-side comparison (demo vs production) for each environment
import matplotlib.pyplot as plt

shift_types = ['gravity', 'obs_noise', 'friction', 'reward_perturb']
shift_labels = {
    'gravity': ('Gravity Scale', 'Gravity Shift'),
    'obs_noise': ('Noise σ', 'Observation Noise'),
    'friction': ('Friction Scale', 'Friction Shift'),
    'reward_perturb': ('Reward Noise σ', 'Reward Perturbation'),
}

for ENV_NAME in ENV_NAMES:
    if ENV_NAME not in PRECOMPUTED or ENV_NAME not in ALL_RESULTS:
        continue
    env_short = ENV_NAME.split('-')[0]
    r = ALL_RESULTS[ENV_NAME]
    pc = PRECOMPUTED[ENV_NAME]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for idx, shift_type in enumerate(shift_types):
        ax = axes[idx // 2, idx % 2]
        xlabel, title = shift_labels[shift_type]

        demo_2q = sorted([x for x in r['results_2q'] if x['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        demo_3q = sorted([x for x in r['results_3q'] if x['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        prod_2q = sorted([x for x in pc['2q'] if x['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])
        prod_3q = sorted([x for x in pc['3q'] if x['shift_type'] == shift_type],
                         key=lambda x: x['shift_level'])

        if demo_2q:
            ax.plot([x['shift_level'] for x in demo_2q], [x['raw_return'] for x in demo_2q],
                    ':o', label='2Q (demo)', color='steelblue', alpha=0.5, linewidth=1.5, markersize=6)
        if demo_3q:
            ax.plot([x['shift_level'] for x in demo_3q], [x['raw_return'] for x in demo_3q],
                    ':s', label='3Q (demo)', color='coral', alpha=0.5, linewidth=1.5, markersize=6)
        if prod_2q:
            ax.plot([x['shift_level'] for x in prod_2q], [x['raw_return'] for x in prod_2q],
                    '-o', label='2Q (300k)', color='steelblue', linewidth=2.5, markersize=8)
        if prod_3q:
            ax.plot([x['shift_level'] for x in prod_3q], [x['raw_return'] for x in prod_3q],
                    '--s', label='3Q (300k)', color='coral', linewidth=2.5, markersize=8)

        ax.set_xlabel(xlabel, fontsize=11)
        ax.set_ylabel('Average Return', fontsize=11)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Demo vs Production — {ENV_NAME}', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'demo_outputs/plots/demo_vs_production_{env_short}.png', dpi=150, bbox_inches='tight')
    print(f'📈 Saved: demo_outputs/plots/demo_vs_production_{env_short}.png')
    plt.show()

if not PRECOMPUTED:
    print('Skipping comparison plots — no pre-computed results available.')

## Step 7: Summary & Export

In [ ]:
# Cell 36: Summary table + AUDC computation for ALL environments
import numpy as np

shift_types = ['gravity', 'obs_noise', 'friction', 'reward_perturb']
BASELINE_LEVELS = {'gravity': 1.0, 'obs_noise': 0.0, 'friction': 1.0, 'reward_perturb': 0.0}
_trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

for ENV_NAME in ENV_NAMES:
    if ENV_NAME not in ALL_RESULTS:
        continue
    r = ALL_RESULTS[ENV_NAME]
    results_2q, results_3q = r['results_2q'], r['results_3q']

    print(f"\n{'='*80}")
    print(f"DEMO RESULTS SUMMARY — {ENV_NAME}")
    print(f"Training: {DEMO_STEPS:,} steps | Seed: {SEED}")
    print(f"{'='*80}")
    print(f"{'Shift Type':<18} {'Level':<8} {'2Q Return':<12} {'2Q Δ(δ)':<10} {'3Q Return':<12} {'3Q Δ(δ)':<10}")
    print(f"{'-'*70}")

    for shift_type in shift_types:
        d2q = sorted([x for x in results_2q if x['shift_type'] == shift_type],
                     key=lambda x: x['shift_level'])
        d3q = sorted([x for x in results_3q if x['shift_type'] == shift_type],
                     key=lambda x: x['shift_level'])
        for r2, r3 in zip(d2q, d3q):
            marker = ' ← baseline' if r2['shift_level'] == BASELINE_LEVELS.get(shift_type) else ''
            print(f"{shift_type:<18} {r2['shift_level']:<8.2f} "
                  f"{r2['raw_return']:<12.2f} {r2['robustness_drop']:<10.4f} "
                  f"{r3['raw_return']:<12.2f} {r3['robustness_drop']:<10.4f}{marker}")
        print()

    # AUDC
    print(f"{'Shift Type':<18} {'2Q AUDC':<12} {'3Q AUDC':<12} {'Winner':<10}")
    print(f"{'-'*50}")
    for shift_type in shift_types:
        d2q = sorted([x for x in results_2q if x['shift_type'] == shift_type],
                     key=lambda x: x['shift_level'])
        d3q = sorted([x for x in results_3q if x['shift_type'] == shift_type],
                     key=lambda x: x['shift_level'])
        audc_2q = _trapz([abs(x['robustness_drop']) for x in d2q],
                         [x['shift_level'] for x in d2q]) if len(d2q) >= 2 else 0
        audc_3q = _trapz([abs(x['robustness_drop']) for x in d3q],
                         [x['shift_level'] for x in d3q]) if len(d3q) >= 2 else 0
        winner = '2Q' if audc_2q < audc_3q else '3Q' if audc_3q < audc_2q else 'Tie'
        print(f"{shift_type:<18} {audc_2q:<12.4f} {audc_3q:<12.4f} {winner:<10}")

print(f"\n{'='*80}")
print('All CSVs were saved during training (see demo_outputs/ directory).')

##  Key Takeaways

1. **IQL trains effectively from offline data** — no environment interaction needed during training
2. **Distribution shift degrades performance** — gravity, friction, noise, and reward perturbations all reduce returns
3. **TripleCritic (3Q) provides more conservative Q-estimates** — via `min(Q₁, Q₂, Q₃)` vs `min(Q₁, Q₂)`
4. **Robustness varies by shift type** — some perturbations are more damaging than others
5. **The expectile τ controls optimism** — higher τ = more optimistic value estimates

### Architecture Insight
```
DoubleCritic: Q(s,a) = min(Q₁, Q₂)        — standard IQL
TripleCritic: Q(s,a) = min(Q₁, Q₂, Q₃)    — our extension (more conservative)
```

In [ ]:
# Download all demo outputs
import os

if IN_COLAB:
    !zip -r demo_outputs.zip demo_outputs/
    from google.colab import files
    files.download('demo_outputs.zip')
    print(" Downloaded demo_outputs.zip — contains all plots, videos, and CSVs")
else:
    print("All outputs saved to demo_outputs/ directory")
    print("Contents:")
    for root, dirs, filenames in os.walk('demo_outputs'):
        for f in filenames:
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024
            print(f"  {path} ({size:.1f} KB)")